In [ ]:
"""
Генерация синтетических данных A/B-эксперимента.

Сценарий: тестируем новую кнопку.
Группа A — контроль, конверсия 10.0%
Группа B — тест, конверсия 11.2% (реальный аплифт +1.2 п.п.)

Результат: data/ab_test.csv с колонками
    user_id, group, converted, revenue
"""

import os
import numpy as np
import pandas as pd


# 1. Воспроизводимость

np.random.seed(42)

# 2. Параметры эксперимента

N_USERS = 60000              # всего пользователей
SHARE_B = 0.5                # доля пользователей в группе B

P_A = 0.100                  # конверсия в группе A (контроль)
P_B = 0.112                  # конверсия в группе B (тест, +1.2 п.п.)

# Распределение выручки от одного конвертированного пользователя.
# Логнормальное распределение — типично для денежных величин:
# основная масса небольших чеков + «тяжёлый хвост» крупных покупок.
REVENUE_MEAN = 1500.0        # средний чек, руб.
REVENUE_SIGMA = 0.6          # разброс (в лог-шкале)


# 3. Формируем пользователей и распределяем по группам

user_ids = np.arange(1, N_USERS + 1)

# Случайно (с заданной вероятностью) относим каждого пользователя
# в группу B, остальные попадают в A.
is_b = np.random.rand(N_USERS) < SHARE_B

groups = np.where(is_b, "B", "A")


# 4. Моделируем конверсию

# Для каждого пользователя бросаем «монетку» с вероятностью,
# зависящей от его группы.
p_convert = np.where(is_b, P_B, P_A)
converted = (np.random.rand(N_USERS) < p_convert).astype(int)

# 5. Моделируем выручку

# Выручка есть только у конвертированных пользователей.
# Логнормальное распределение: revenue = exp(mu + sigma * z),
# где mu подобран так, чтобы среднее равнялось REVENUE_MEAN.
mu = np.log(REVENUE_MEAN) - 0.5 * REVENUE_SIGMA ** 2

revenue = np.zeros(N_USERS)
n_converted = converted.sum()
revenue[converted == 1] = np.random.lognormal(
    mean=mu,
    sigma=REVENUE_SIGMA,
    size=n_converted,
).round(2)

# 6. Собираем DataFrame

df = pd.DataFrame({
    "user_id": user_ids,
    "group": groups,
    "converted": converted,
    "revenue": revenue,
})


# 7. Сохраняем в CSV

os.makedirs("data", exist_ok=True)
df.to_csv("data/ab_test.csv", index=False)


# 8. Краткая сводка по сгенерированным данным
print("Сгенерировано пользователей:", len(df))
print()
summary = (
    df.groupby("group")
      .agg(
          users=("user_id", "count"),
          conversions=("converted", "sum"),
          conversion_rate=("converted", "mean"),
          total_revenue=("revenue", "sum"),
          arpu=("revenue", "mean"),
      )
      .round(4)
)
print(summary)
print()

lift = (
    summary.loc["B", "conversion_rate"]
    - summary.loc["A", "conversion_rate"]
)
print(f"Абсолютный аплифт конверсии (B - A): {lift:+.4f}")
print(f"Относительный аплифт: {lift / summary.loc['A', 'conversion_rate']:+.2%}")
print()
print("Файл сохранён: data/ab_test.csv")

Сгенерировано пользователей: 60000

       users  conversions  conversion_rate  total_revenue      arpu
group                                                              
A      30009         2959           0.0986     4386264.55  146.1650
B      29991         3422           0.1141     5112937.62  170.4824

Абсолютный аплифт конверсии (B - A): +0.0155
Относительный аплифт: +15.72%

Файл сохранён: data/ab_test.csv


In [ ]:
"""
Анализ A/B-теста новой кнопки.

Читает data/ab_test.csv, считает всю статистику, печатает ключевые цифры,
сохраняет график в outputs/ и числовой отчёт в outputs/results.txt.

Используются только pandas / numpy / matplotlib + стандартная библиотека.
Стат-функции (z-тест, хи-квадрат, нормальное распределение) реализованы вручную,
чтобы не зависеть от scipy.
"""

import os
import math
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # без дисплея, рендерим в файл
import matplotlib.pyplot as plt

HERE = "/content"
DATA_DIR = os.path.join(HERE, "data")
OUT_DIR = os.path.join(HERE, "outputs")
ALPHA = 0.05


# ---------- вспомогательные стат-функции (без scipy) ----------

def norm_cdf(x):
    """Функция распределения стандартного нормального через erf."""
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def two_sided_p_from_z(z):
    """Двусторонний p-value для z-статистики."""
    return 2.0 * (1.0 - norm_cdf(abs(z)))


def chi2_sf_df1(x):
    """
    P(Chi2_1 > x) для 1 степени свободы.
    Для df=1 хи-квадрат = z^2, поэтому хвост = 2*(1 - Phi(sqrt(x))).
    """
    if x <= 0:
        return 1.0
    return 2.0 * (1.0 - norm_cdf(math.sqrt(x)))


# ---------- основной анализ ----------

def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    df = pd.read_csv(os.path.join(DATA_DIR, "ab_test.csv"))

    a = df[df["group"] == "A"]
    b = df[df["group"] == "B"]

    n_a, n_b = len(a), len(b)
    conv_a, conv_b = int(a["converted"].sum()), int(b["converted"].sum())
    p_a, p_b = conv_a / n_a, conv_b / n_b

    # --- Sample Ratio Mismatch: ожидаем 50/50, проверяем хи-квадратом ---
    n_total = n_a + n_b
    exp = n_total / 2.0
    srm_chi2 = (n_a - exp) ** 2 / exp + (n_b - exp) ** 2 / exp
    srm_p = chi2_sf_df1(srm_chi2)
    srm_ok = srm_p >= 0.05  # ok, если НЕ значимо (распределение здоровое)

    # --- Z-тест разницы двух пропорций (pooled) ---
    p_pool = (conv_a + conv_b) / (n_a + n_b)
    se_pool = math.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
    diff = p_b - p_a
    z = diff / se_pool
    p_value = two_sided_p_from_z(z)

    # --- Хи-квадрат на таблице 2x2 (с поправкой Йейтса) — для сверки ---
    conv = np.array([[conv_a, n_a - conv_a],
                     [conv_b, n_b - conv_b]], dtype=float)
    row = conv.sum(axis=1)
    col = conv.sum(axis=0)
    expected = np.outer(row, col) / n_total
    chi2_stat = ((np.abs(conv - expected) - 0.5) ** 2 / expected).sum()
    chi2_p = chi2_sf_df1(chi2_stat)

    # --- 95% доверительный интервал разницы (unpooled SE) ---
    se_unpooled = math.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
    ci_low = diff - 1.96 * se_unpooled
    ci_high = diff + 1.96 * se_unpooled

    # --- Аплифты ---
    abs_uplift = diff
    rel_uplift = diff / p_a

    significant = p_value < ALPHA

    # --- ДИ для каждой группы (для графика, Wald) ---
    ci_a = 1.96 * math.sqrt(p_a * (1 - p_a) / n_a)
    ci_b = 1.96 * math.sqrt(p_b * (1 - p_b) / n_b)

    # ---------- печать ----------
    lines = []

    def out(s=""):
        print(s)
        lines.append(s)

    out("=" * 56)
    out("A/B-ТЕСТ: новая кнопка повышает конверсию?")
    out("=" * 56)
    out()
    out(f"Группа A: {n_a:>6} польз., {conv_a:>5} конверсий, CR = {p_a*100:.2f}%")
    out(f"Группа B: {n_b:>6} польз., {conv_b:>5} конверсий, CR = {p_b*100:.2f}%")
    out()
    out("--- Sample Ratio Mismatch (ожидали 50/50) ---")
    out(f"chi2 = {srm_chi2:.3f}, p = {srm_p:.4f} -> "
        f"{'распределение здоровое (OK)' if srm_ok else 'ВНИМАНИЕ: перекос групп!'}")
    out()
    out("--- Разница конверсий ---")
    out(f"Абсолютный аплифт: {abs_uplift*100:+.2f} п.п.")
    out(f"Относительный аплифт: {rel_uplift*100:+.2f}%")
    out(f"95% ДИ разницы: [{ci_low*100:+.2f} п.п.; {ci_high*100:+.2f} п.п.]")
    out()
    out("--- Проверка значимости ---")
    out(f"Z-тест двух пропорций: z = {z:.3f}, p-value = {p_value:.5f}")
    out(f"Хи-квадрат 2x2 (Йейтс): chi2 = {chi2_stat:.3f}, p-value = {chi2_p:.5f}")
    out(f"Уровень значимости alpha = {ALPHA}")
    out()
    if significant:
        out(f"ВЫВОД: разница СТАТИСТИЧЕСКИ ЗНАЧИМА (p = {p_value:.5f} < {ALPHA}).")
        out("Новая кнопка реально повышает конверсию — можно раскатывать на всех.")
    else:
        out(f"ВЫВОД: разница НЕ значима (p = {p_value:.5f} >= {ALPHA}).")
        out("Данных недостаточно, чтобы утверждать эффект. Не раскатываем.")
    out("=" * 56)

    # ---------- график ----------
    fig, ax = plt.subplots(figsize=(7, 5))
    groups = ["A (старая)", "B (новая)"]
    means = [p_a * 100, p_b * 100]
    errs = [ci_a * 100, ci_b * 100]
    colors = ["#9aa7b1", "#e0a458"]

    bars = ax.bar(groups, means, yerr=errs, capsize=10,
                  color=colors, edgecolor="#444", linewidth=1.2,
                  error_kw={"elinewidth": 1.5, "ecolor": "#444"})

    for bar, m in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, m + max(errs) * 0.15,
                f"{m:.2f}%", ha="center", va="bottom",
                fontsize=12, fontweight="bold")

    ax.set_ylabel("Конверсия, %")
    ax.set_title("Конверсия по группам (95% ДИ)\n"
                 f"абс. аплифт {abs_uplift*100:+.2f} п.п., p = {p_value:.4f}")
    ax.set_ylim(0, max(means) + max(errs) * 2.2)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()

    chart_path = os.path.join(OUT_DIR, "conversion_by_group.png")
    fig.savefig(chart_path, dpi=130)
    plt.close(fig)
    out()
    out(f"График сохранён: {chart_path}")

    # ---------- results.txt ----------
    res_path = os.path.join(OUT_DIR, "results.txt")
    with open(res_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")
    print(f"Отчёт сохранён: {res_path}")

    # вернём числа для README/meta
    return {
        "n_a": n_a, "n_b": n_b,
        "p_a": p_a, "p_b": p_b,
        "srm_chi2": srm_chi2, "srm_p": srm_p,
        "z": z, "p_value": p_value,
        "ci_low": ci_low, "ci_high": ci_high,
        "abs_uplift": abs_uplift, "rel_uplift": rel_uplift,
        "significant": significant,
    }


if __name__ == "__main__":
    main()

A/B-ТЕСТ: новая кнопка повышает конверсию?

Группа A:  30009 польз.,  2959 конверсий, CR = 9.86%
Группа B:  29991 польз.,  3422 конверсий, CR = 11.41%

--- Sample Ratio Mismatch (ожидали 50/50) ---
chi2 = 0.005, p = 0.9414 -> распределение здоровое (OK)

--- Разница конверсий ---
Абсолютный аплифт: +1.55 п.п.
Относительный аплифт: +15.72%
95% ДИ разницы: [+1.06 п.п.; +2.04 п.п.]

--- Проверка значимости ---
Z-тест двух пропорций: z = 6.157, p-value = 0.00000
Хи-квадрат 2x2 (Йейтс): chi2 = 37.742, p-value = 0.00000
Уровень значимости alpha = 0.05

ВЫВОД: разница СТАТИСТИЧЕСКИ ЗНАЧИМА (p = 0.00000 < 0.05).
Новая кнопка реально повышает конверсию — можно раскатывать на всех.

График сохранён: /content/outputs/conversion_by_group.png
Отчёт сохранён: /content/outputs/results.txt
